# Continue your saved R3GAN model to 13 total training hours

Connect a GPU runtime. Stop the original training run before continuing. Run sections in order, **not Run all**. This notebook reuses the original software ZIP and prepared dataset on Drive. No image ZIP upload is needed when that dataset is still present.

The latest verified full checkpoint is copied into `neohuman-colab-001-13h` with a larger budget. The learned weights, EMA, optimizers, random states, sampler, event timers and elapsed training time are preserved. **13 hours is the total, not 13 extra hours.** The original experiment remains a backup; continue training only the `-13h` experiment afterward. A checkpoint with 8 saved hours has about 5 hours remaining.

An interrupted runtime may leave a writer lock. This notebook does not delete locks automatically. If the extension cell reports a stale lock, use the original notebook's explicit lock-inspection/release procedure only after confirming every old writer has stopped, then retry. Re-running the extension cell reuses a completed continuation and never resets its progress. An incomplete failed extension is refused; preserve it and select another destination ID.

Test evidence and limits are recorded in VALIDATION.md. Live Colab execution and a real 13-hour session have not been tested.


## 1 · Settings and GPU check
Keep `SOURCE_ID` as the original run. On later sessions keep all these settings the same.


In [ ]:
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/NeoHuman_R3GAN")
SCRATCH = Path("/content/NeoHuman_R3GAN_extend_work")
CODE_ROOT = Path("/content/NeoHuman_R3GAN_extend_code")
CODE_BUNDLE = PROJECT / "software" / "NeoHuman_R3GAN_Workspace.zip"
SOURCE_ID = "neohuman-colab-001"
CONTINUATION_ID = "neohuman-colab-001-13h"
TOTAL_HOURS = 13


In [ ]:
import platform, shutil, subprocess, sys
print("OS:", platform.platform(), "Python:", sys.version)
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)
else:
    raise RuntimeError("No NVIDIA runtime found. Select a Colab GPU or install/check the laptop NVIDIA driver before continuing.")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Use Python 3.11–3.13 for these pins. Local instructions use Python 3.11.")


## 2 · Mount Drive and load the original pinned software
If installation requests a restart, restart and rerun from section 1.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PROJECT.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)
print("Persistent project:", PROJECT)


In [ ]:
import hashlib, importlib, json, os, zipfile
from pathlib import PurePosixPath
if not CODE_BUNDLE.exists():
    from google.colab import files
    print("Upload NeoHuman_R3GAN_Workspace.zip (the SOFTWARE package)")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one workspace ZIP")
    payload = next(iter(uploaded.values()))
    CODE_BUNDLE.parent.mkdir(parents=True, exist_ok=True)
    temporary = CODE_BUNDLE.with_suffix(".partial")
    temporary.write_bytes(payload)
    os.replace(temporary, CODE_BUNDLE)
with zipfile.ZipFile(CODE_BUNDLE) as package:
    inventory = json.loads(package.read("workspace-package.json"))["files"]
    if set(package.namelist()) != set(inventory) | {"workspace-package.json"}:
        raise ValueError("Workspace ZIP inventory mismatch")
    for name, expected in inventory.items():
        p = PurePosixPath(name)
        if p.is_absolute() or ".." in p.parts or "\\" in name or ":" in name:
            raise ValueError("Unsafe package path")
        data = package.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError("Workspace ZIP checksum mismatch")
        target = CODE_ROOT / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
loaded = {name: str(getattr(sys.modules[name], "__version__", "")) for name in ("torch", "numpy", "PIL") if name in sys.modules}
subprocess.run([sys.executable, "-m", "pip", "install", "pip==25.1.1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "matplotlib==3.10.3"], check=True)
subprocess.run([sys.executable, "scripts/fetch_upstream.py"], check=True)
for name, pin in {"torch": "2.7.1", "numpy": "2.2.6", "PIL": "11.2.1"}.items():
    if name in loaded and loaded[name].split("+")[0] != pin:
        raise RuntimeError("Dependencies changed. Restart the Colab session, then rerun this continuation notebook from section 1.")
importlib.invalidate_caches()


## 3 · Define the full-state extension helper
This keeps the shared training code unchanged so the old checkpoint remains compatible.


In [ ]:
"""Create a full-state continuation with a larger total budget; preserve the source.

This lives outside neo_r3gan so existing checkpoint code identities stay valid.
Only the experiment ID, total budget and corresponding config hash change.
"""
import copy
import math
import tempfile
import time
from pathlib import Path

from neo_r3gan.checkpoints import Checkpoints, RunLock
from neo_r3gan.config import config_hash, default_config, load_config, validate
from neo_r3gan.data import verify_prepared
from neo_r3gan.io_utils import digest, read_json, safe_child, write_json
from neo_r3gan.upstream import code_version, require_pins


def extend_training_budget(project, scratch, source_id, destination_id, total_hours=13):
    """Source must be stopped. Repeated calls reuse a verified continuation."""
    for name in (source_id, destination_id):
        validate(default_config(name))
    if source_id == destination_id:
        raise ValueError("Use a different continuation ID to preserve the source")
    if isinstance(total_hours, bool) or not isinstance(total_hours, (int, float)) or not math.isfinite(total_hours) or total_hours <= 0:
        raise ValueError("Total hours must be finite and positive")
    require_pins()
    project, scratch = Path(project), Path(scratch)
    source = project / "experiments" / source_id
    destination = project / "experiments" / destination_id
    if not (source / "run.json").is_file():
        raise FileNotFoundError("Saved source experiment not found; check Drive and source ID")
    scratch.mkdir(parents=True, exist_ok=True)
    with RunLock(source), tempfile.TemporaryDirectory(dir=scratch, prefix="extend-budget-") as temporary:
        local = Path(temporary)
        if read_json(source / "run.json")["status"] != "active":
            raise RuntimeError("Source is sealed for transfer; use the imported active run")
        original = load_config(source / "config.json")
        if original["experiment_id"] != source_id:
            raise ValueError("Source configuration belongs to another experiment")
        target_seconds = total_hours * 3600
        if target_seconds <= original["schedule"]["budget_seconds"]:
            raise ValueError("The new total budget must exceed the source budget")
        version = code_version()
        if read_json(source / "code_version.json") != version:
            raise ValueError("Use the original software package for this checkpoint")
        dataset = read_json(source / "dataset_manifest.json")
        prepared = safe_child(project / "datasets" / "prepared", dataset["dataset_id"])
        if verify_prepared(prepared) != dataset:
            raise ValueError("Prepared dataset differs from the source experiment")
        extended = copy.deepcopy(original)
        extended["experiment_id"] = destination_id
        extended["schedule"]["budget_seconds"] = target_seconds
        validate(extended)
        identity = {"source_id": source_id, "source_config_sha256": config_hash(original),
                    "destination_id": destination_id, "total_budget_seconds": target_seconds}
        if destination.exists():
            # Never overwrite a previous attempt, another experiment or a
            # continuation that has already advanced beyond the fork point.
            with RunLock(destination):
                if not (destination / "run.json").is_file() or not (destination / "extension.json").is_file():
                    raise RuntimeError("Destination exists but extension is incomplete; preserve it and use a new destination ID")
                if read_json(destination / "run.json")["status"] != "active":
                    raise RuntimeError("Continuation is sealed for transfer")
                provenance = read_json(destination / "extension.json")
                if any(provenance.get(k) != v for k, v in identity.items()) or read_json(destination / "config.json") != extended:
                    raise ValueError("Destination belongs to a different extension; refusing to overwrite it")
                if read_json(destination / "dataset_manifest.json") != dataset or read_json(destination / "code_version.json") != version:
                    raise ValueError("Continuation dataset/code metadata mismatch")
                state, entry = Checkpoints(destination, local).load_latest()
                if state["config"] != extended or state["config_sha256"] != config_hash(extended) or state["code"] != version or state["dataset_sha256"] != digest(dataset):
                    raise ValueError("Continuation checkpoint identity mismatch")
                del state
                print(f"Using existing continuation at {entry['training_seconds']/3600:.4f} h; remaining {max(0, target_seconds-entry['training_seconds'])/3600:.4f} h.")
            return extended, prepared
        state, entry = Checkpoints(source, local).load_latest()
        if state["schema"] != 1 or state["code"] != version or state["config"] != original or state["config_sha256"] != config_hash(original) or state["dataset_sha256"] != digest(dataset):
            raise ValueError("Source checkpoint configuration/dataset/code mismatch")
        if any(state["counters"][key] != entry[key] for key in ("step", "training_seconds")):
            raise ValueError("Checkpoint counters differ from its manifest")
        if target_seconds <= entry["training_seconds"]:
            raise ValueError("New total budget is already exhausted by the saved checkpoint")
        # The models, EMA, optimizers, counters, RNGs, sampler and event timers
        # are intentionally untouched. No trainer or new weights are created.
        state["config"] = extended
        state["config_sha256"] = config_hash(extended)
        destination.mkdir(parents=True, exist_ok=False)
        with RunLock(destination) as lock:
            for name, value in (("config.json", extended), ("dataset_manifest.json", dataset),
                                ("code_version.json", version), ("environment.json", state["environment"])):
                write_json(destination / name, value)
            manager = Checkpoints(destination, local / "destination", extended["schedule"]["keep_recovery"], lock)
            saved = manager.save(state, ("recovery", "manual"))
            checked, _ = manager.load_latest()
            from neo_r3gan.workflow import _compare
            _compare(state, checked)  # Read back every tensor and saved state field.
            del checked, state
            write_json(destination / "extension.json", dict(identity, source_checkpoint=entry, created=time.time()))
            write_json(project / "configs" / (destination_id + ".json"), extended)
            # Published last: incomplete attempts are never treated as usable.
            write_json(destination / "run.json", {"status": "active", "experiment_id": destination_id, "created": time.time(), "continued_from": source_id})
            print(f"Verified continuation at step {saved['step']}, {saved['training_seconds']/3600:.4f} saved hours; remaining {(target_seconds-saved['training_seconds'])/3600:.4f} h of {total_hours} total.")
    return extended, prepared


## 4 · Create or reopen the 13-hour continuation
The source training run must be stopped. This verifies the copied checkpoint before making the continuation resumable. A second run of this cell reuses the continuation's current checkpoint.


In [ ]:
CFG, PREPARED = extend_training_budget(PROJECT, SCRATCH, SOURCE_ID, CONTINUATION_ID, TOTAL_HOURS)
print("Continue this experiment:", CFG["experiment_id"])
print("Total training-hour limit:", CFG["schedule"]["budget_seconds"] / 3600)


## 5 · Run the GPU save/resume test
Required after each runtime restart. This uses a separate smoke-test experiment.


In [ ]:
from neo_r3gan.workflow import smoke_test, train, latest_verified, clear_stop, show_progress, generate_images
from IPython.display import display
from PIL import Image
SMOKE_REPORT = smoke_test(CFG, PROJECT, SCRATCH, PREPARED)
display(Image.open(SMOKE_REPORT["preview"]))


## 6 · Resume latest verified checkpoint
Set `RESUME_RUN = True`, then run this cell. It restores the saved elapsed time and stops when the cumulative total reaches 13 hours. Keep the saved configuration unchanged. GPU quota and session limits still apply; saved progress can resume in a later GPU session.


In [ ]:
RESUME_RUN = False  # Change to True when ready to continue.
if RESUME_RUN:
    latest_verified(PROJECT, CFG)
    clear_stop(PROJECT, CFG)
    RESULT = train(CFG, PROJECT, SCRATCH, PREPARED, resume=True)
else:
    print("Set RESUME_RUN=True to continue the saved model to 13 total hours.")


## 7 · View results after training stops
The extension run writes new samples and checkpoints under its `-13h` folder. The original run's older samples remain in its original folder.


In [ ]:
RUN_FOLDER = show_progress(PROJECT, CFG)
print(RUN_FOLDER)


### Generate individual images on GPU
For CPU inference later, use the CPU notebook with `EXPERIMENT_ID = "neohuman-colab-001-13h"`.


In [ ]:
from neo_r3gan.data import grid
GENERATED = generate_images(PROJECT, SCRATCH, CFG, PREPARED, seeds=range(16))
display(grid([Image.open(path) for path in GENERATED], columns=4))
print("Saved:", GENERATED[0].parent)
